### Get olama

In [ ]:
# 1. Install the missing dependency (zstd) and pciutils (for GPU detection)
!sudo apt-get update
!sudo apt-get install -y zstd pciutils

# 2. Install Ollama
# !curl -fsSL https://ollama.com/install.sh | sh

# 3. Launch Ollama Server in the background
import subprocess
import time

# Start the server
# process = subprocess.Popen(['ollama', 'serve'])

# Give it 10 seconds to fully initialize
# time.sleep(10)

# 4. Pull your model
# !ollama pull llama3.2

: 

### Environment Setup
Install the necessary libraries and start the Ollama server if you are on Google Colab.

In [ ]:
# Install dependencies
# First, uninstall conflicting packages to avoid dependency resolution errors
!pip uninstall -y google-adk opentelemetry-exporter-gcp-logging
!pip install pydantic-ai aiohttp pydantic

# For Colab users: Install and start Ollama (if not already running)
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time

# Ensure the Ollama server is started in the background
# It's important to run this in the background and give it time to initialize.
# We will use a try-except block to handle cases where it might already be running
try:
    subprocess.Popen(['ollama', 'serve'])
    print("Ollama server started.")
except Exception as e:
    print(f"Could not start Ollama server (it might already be running): {e}")

# Give it ample time to fully initialize. Increased from 5 to 15 seconds for robustness.
time.sleep(15)

# Pull the desired model. This needs to complete successfully.
print("Pulling llama3.2:3b model...")
!ollama pull llama3.2:3b
print("Model pull complete.")

Found existing installation: google-adk 1.29.0
Uninstalling google-adk-1.29.0:
  Successfully uninstalled google-adk-1.29.0
Found existing installation: opentelemetry-exporter-gcp-logging 1.11.0a0
Uninstalling opentelemetry-exporter-gcp-logging-1.11.0a0:
  Successfully uninstalled opentelemetry-exporter-gcp-logging-1.11.0a0
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/pp

### Imports and Configuration
Keep your imports and the model configuration in their own space.

In [ ]:
import asyncio
import os
import re
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import aiohttp
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

# 2026-style local provider setup
provider = OpenAIProvider(base_url="http://localhost:11434/v1", api_key="ollama")
model = OpenAIChatModel(model_name="llama3.2:3b", provider=provider)

### Data Schemas & Agent Definition
Define what "Success" looks like for your agent.

In [ ]:
class PaperAnalysis(BaseModel):
    title: str
    year: Optional[int] = None
    venue: Optional[str] = None
    url: Optional[str] = None
    doi: Optional[str] = None
    key_points: List[str] = Field(default_factory=list)
    why_relevant: List[str] = Field(default_factory=list)

class ResearchReport(BaseModel):
    query: str
    papers: List[PaperAnalysis]

analysis_agent = Agent(
    model=model,
    system_prompt=(
        "You are a research assistant. You MUST respond with valid JSON "
        "matching the ResearchReport schema. Do not include conversational filler."
    )
)

### Retrieval Logic (The "Tools")
Paste all the helper functions for OpenAlex and Semantic Scholar here. These don't change often, so they belong in a single "logic" cell.

In [ ]:
@dataclass
class RetrievedPaper:
    title: str
    abstract: Optional[str]
    year: Optional[int]
    venue: Optional[str]
    url: Optional[str]
    doi: Optional[str]
    source: str


def _normalize_title(t: str) -> str:
    t = t.lower().strip()
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"[^a-z0-9 ]+", "", t)
    return t

def _dedupe(papers: List[RetrievedPaper]) -> List[RetrievedPaper]:
    seen: set[Tuple[Optional[str], str]] = set()
    out: List[RetrievedPaper] = []
    for p in papers:
        key = (p.doi.lower().strip() if p.doi else None, _normalize_title(p.title))
        if key in seen:
            continue
        seen.add(key)
        out.append(p)
    return out

def _openalex_abstract_from_inverted_index(inv: Optional[Dict[str, List[int]]]) -> Optional[str]:
    # OpenAlex stores abstract as an inverted index. Reconstruct if present.
    if not inv:
        return None
    positions: Dict[int, str] = {}
    for word, pos_list in inv.items():
        for pos in pos_list:
            positions[pos] = word
    if not positions:
        return None
    return " ".join(positions[i] for i in sorted(positions.keys()))

async def search_openalex(session: aiohttp.ClientSession, query: str, per_page: int = 8) -> List[RetrievedPaper]:
    # OpenAlex works search param searches title/abstract/fulltext. :contentReference[oaicite:3]{index=3}
    url = "https://api.openalex.org/works"
    params = {
        "search": query,
        "per-page": str(per_page),
    }
    async with session.get(url, params=params, timeout=30) as r:
        r.raise_for_status()
        data = await r.json()

    results = []
    for item in data.get("results", []):
        title = item.get("display_name") or "Untitled"
        year = item.get("publication_year")
        doi = item.get("doi")
        venue = None
        primary_location = item.get("primary_location") or {}
        source = primary_location.get("source") or {}
        venue = source.get("display_name") or item.get("host_venue", {}).get("display_name")

        abstract = _openalex_abstract_from_inverted_index(item.get("abstract_inverted_index"))

        # best-effort URL
        url_out = item.get("id") or item.get("primary_location", {}).get("landing_page_url")
        results.append(
            RetrievedPaper(
                title=title,
                abstract=abstract,
                year=year,
                venue=venue,
                url=url_out,
                doi=doi,
                source="openalex",
            )
        )
    return results

async def search_semantic_scholar(session: aiohttp.ClientSession, query: str, limit: int = 8) -> List[RetrievedPaper]:
    # Semantic Scholar bulk search endpoint: /graph/v1/paper/search/bulk :contentReference[oaicite:4]{index=4}
    url = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
    params = {
        "query": query,
        "limit": str(limit),
        "fields": "title,abstract,year,venue,url,externalIds",
        # you can add filters like year="2023-" if you want
    }

    headers = {}
    # Optional but recommended: set S2 API key if you have it
    # export S2_API_KEY="..."
    s2_key = os.getenv("S2_API_KEY")
    if s2_key:
        headers["x-api-key"] = s2_key

    async with session.get(url, params=params, headers=headers, timeout=30) as r:
        r.raise_for_status()
        data = await r.json()

    results = []
    for item in data.get("data", []) or []:
        external = item.get("externalIds") or {}
        doi = external.get("DOI")
        results.append(
            RetrievedPaper(
                title=item.get("title") or "Untitled",
                abstract=item.get("abstract"),
                year=item.get("year"),
                venue=item.get("venue"),
                url=item.get("url"),
                doi=(f"https://doi.org/{doi}" if doi and not str(doi).startswith("http") else doi),
                source="semanticscholar",
            )
        )
    return results


### The Orchestrator & UI
This is your "Main" function that connects the tools to the agent.

In [ ]:
def render_markdown(report: ResearchReport) -> str:
    lines = []
    lines.append(f"# Research helper results\n")
    lines.append(f"Query: {report.query}\n")

    for i, p in enumerate(report.papers, start=1):
        meta = []
        if p.year:
            meta.append(str(p.year))
        if p.venue:
            meta.append(p.venue)
        meta_str = " | ".join(meta) if meta else ""

        lines.append(f"## {i}. {p.title}")
        if meta_str:
            lines.append(f"{meta_str}")

        if p.url:
            lines.append(f"Source: {p.url}")
        elif p.doi:
            lines.append(f"Source: {p.doi}")
        else:
            lines.append("Source: (not provided)")

        if p.key_points:
            lines.append("\nKey points:")
            for b in p.key_points:
                lines.append(f"- {b}")

        if p.why_relevant:
            lines.append("\nWhy this matches your query:")
            for b in p.why_relevant:
                lines.append(f"- {b}")

        lines.append("\n")
    return "\n".join(lines)

In [ ]:
import json

async def run_research_helper(user_text: str, k_each: int = 8) -> None:
    # 1. RETRIEVAL (The part that was likely missing)
    async with aiohttp.ClientSession() as session:
        oa_task = search_openalex(session, user_text, per_page=k_each)
        s2_task = search_semantic_scholar(session, user_text, limit=k_each)
        openalex_papers, s2_papers = await asyncio.gather(oa_task, s2_task)

    combined = _dedupe(openalex_papers + s2_papers)
    top = combined[:10]

    # 2. CREATE THE PAYLOAD (Defining payload_items)
    payload_items = []
    for p in top:
        payload_items.append({
            "title": p.title,
            "abstract": p.abstract[:500] if p.abstract else "No abstract provided.",
            "url": p.url or p.doi
        })

    # 3. PROMPT & RUN
    prompt = (
        f"User Query: {user_text}\n\n"
        f"Analyze these papers and return ONLY a JSON object.\n"
        f"Data: {payload_items}"
    )

    result = await analysis_agent.run(prompt)

    # 4. MANUAL PARSING (Safest for Local Models)
    try:
        # Strip markdown code blocks (```json ... ```) if they exist
        raw_output = result.output
        clean_json = re.sub(r"```json|```", "", raw_output).strip()

        # Load into Pydantic model for safety
        report_data = json.loads(clean_json)
        report = ResearchReport.model_validate(report_data)

        print(render_markdown(report))

    except Exception as e:
        print(f"❌ Parsing Error: {e}")
        print("-- RAW RESPONSE FROM LLM --")
        print(result.output)

# Now run the final cell
# await run_research_helper("IoT DDoS detection using lightweight models")

### Run Your Research
Finally, the interactive part! You can change the query in this cell and run it as many times as you want.

In [ ]:
QUERY = "On-device LLM reasoning for IoT DDoS detection"

await run_research_helper(QUERY)

❌ Parsing Error: 2 validation errors for ResearchReport
query
  Field required [type=missing, input_value={'research Papers': [{'ti...rs': [], 'journal': {}}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
papers
  Field required [type=missing, input_value={'research Papers': [{'ti...rs': [], 'journal': {}}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
-- RAW RESPONSE FROM LLM --
{
  "research Papers": [
    {
      "title": "Application ofLarge Language Models toDDoS Attack Detection",
      "authors": [],
      "journal": null,
      "year": 2024,
      "source": "https://openalex.org/W4391519594"
    },
    {
      "title": "Rethinking On-Device LLM Reasoning: Why Analogical Mapping Outperforms Abstract Thinking for IoT DDoS Detection",
      "authors": [],
      "journal": null,
      "year": 2023,
      "source": [
        "https://openalex.org/W7125406502",
        "https://openalex.